# Similarity Search & Approximate Nearest Neighbors

**Core interview topic**: How do you find similar items at scale?

Covers:
1. Brute-force similarity search (cosine, euclidean)
2. Locality-Sensitive Hashing (LSH) for cosine similarity
3. Product quantization (vector compression + approximate distance)
4. Approximate nearest neighbors with random projections
5. FAISS patterns overview (IVF, HNSW)
6. Benchmarks: exact vs approximate on 10K vectors

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from collections import defaultdict

np.random.seed(42)

## 1. Brute-Force Similarity Search

**Cosine similarity**: $\text{sim}(a,b) = \frac{a \cdot b}{\|a\|\|b\|}$

**Euclidean distance**: $d(a,b) = \|a - b\|_2$

Brute force is O(Nd) per query — fine for small N, impractical at scale.

In [ ]:
def cosine_similarity(a, b):
    """Cosine similarity between two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10)


def euclidean_distance(a, b):
    """L2 distance between two vectors."""
    return np.linalg.norm(a - b)


def brute_force_search(database, query, k=5, metric='cosine'):
    """
    Brute-force k-NN search.
    database: (N, d) array
    query: (d,) array
    Returns: indices, scores for top-k
    """
    if metric == 'cosine':
        # Vectorized cosine similarity against entire database
        norms = np.linalg.norm(database, axis=1) + 1e-10
        q_norm = np.linalg.norm(query) + 1e-10
        scores = database @ query / (norms * q_norm)
        top_k = np.argsort(scores)[-k:][::-1]  # highest similarity
        return top_k, scores[top_k]
    elif metric == 'euclidean':
        # Vectorized L2 distance
        dists = np.linalg.norm(database - query[None, :], axis=1)
        top_k = np.argsort(dists)[:k]  # smallest distance
        return top_k, dists[top_k]
    else:
        raise ValueError(f"Unknown metric: {metric}")


# Quick demo
N, d = 1000, 128
db = np.random.randn(N, d).astype(np.float32)
query = np.random.randn(d).astype(np.float32)

idx_cos, scores_cos = brute_force_search(db, query, k=5, metric='cosine')
idx_euc, dists_euc = brute_force_search(db, query, k=5, metric='euclidean')

print("Cosine top-5 indices:", idx_cos, "scores:", np.round(scores_cos, 4))
print("Euclidean top-5 indices:", idx_euc, "dists:", np.round(dists_euc, 4))

## 2. Locality-Sensitive Hashing (LSH) for Cosine Similarity

**Key idea**: Hash vectors so that similar vectors land in the same bucket with high probability.

For cosine similarity, use **random hyperplane hashing**:
- Sample random vector $r \sim \mathcal{N}(0, I)$
- Hash bit: $h(x) = \text{sign}(r \cdot x)$
- Probability of collision: $P[h(a)=h(b)] = 1 - \frac{\theta(a,b)}{\pi}$ where $\theta$ is the angle between $a$ and $b$

Use multiple hash functions + multiple tables to boost recall.

In [ ]:
class LSHCosine:
    """
    Locality-Sensitive Hashing for cosine similarity.
    Uses random hyperplane hashing with multiple hash tables.
    """
    def __init__(self, dim, num_bits=8, num_tables=4):
        """
        dim: vector dimension
        num_bits: number of hash bits per table (more bits = fewer collisions = lower recall, higher precision)
        num_tables: number of independent hash tables (more tables = higher recall)
        """
        self.dim = dim
        self.num_bits = num_bits
        self.num_tables = num_tables
        
        # Random hyperplanes: each table has num_bits random hyperplanes
        self.hyperplanes = [
            np.random.randn(num_bits, dim).astype(np.float32)
            for _ in range(num_tables)
        ]
        
        # Hash tables: table_idx -> {hash_key: [vector_indices]}
        self.tables = [defaultdict(list) for _ in range(num_tables)]
        self.data = None
    
    def _hash(self, vec, table_idx):
        """Compute hash key for a vector using hyperplane projections."""
        projections = self.hyperplanes[table_idx] @ vec  # (num_bits,)
        bits = (projections > 0).astype(int)
        # Convert bit array to integer key
        return tuple(bits.tolist())
    
    def index(self, data):
        """Build the index from data array (N, d)."""
        self.data = data
        for table_idx in range(self.num_tables):
            for i in range(len(data)):
                key = self._hash(data[i], table_idx)
                self.tables[table_idx][key].append(i)
    
    def query(self, q, k=5):
        """
        Find approximate nearest neighbors.
        Returns: indices, cosine similarities
        """
        # Step 1: collect candidates from all tables
        candidates = set()
        for table_idx in range(self.num_tables):
            key = self._hash(q, table_idx)
            candidates.update(self.tables[table_idx].get(key, []))
        
        if len(candidates) == 0:
            return np.array([]), np.array([])
        
        # Step 2: re-rank candidates with exact cosine similarity
        candidates = list(candidates)
        candidate_vecs = self.data[candidates]
        norms = np.linalg.norm(candidate_vecs, axis=1) + 1e-10
        q_norm = np.linalg.norm(q) + 1e-10
        sims = candidate_vecs @ q / (norms * q_norm)
        
        top_k_local = np.argsort(sims)[-k:][::-1]
        result_idx = np.array(candidates)[top_k_local]
        result_sims = sims[top_k_local]
        return result_idx, result_sims


# Build and test
lsh = LSHCosine(dim=d, num_bits=10, num_tables=5)
lsh.index(db)

lsh_idx, lsh_sims = lsh.query(query, k=5)
print("LSH top-5 indices:", lsh_idx, "sims:", np.round(lsh_sims, 4))
print("Exact top-5 indices:", idx_cos)
print("Overlap:", len(set(lsh_idx) & set(idx_cos)), "/ 5")

### Visualize: LSH Hash Bucket Distribution

In [ ]:
# Visualize bucket sizes for one hash table
bucket_sizes = [len(v) for v in lsh.tables[0].values()]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(bucket_sizes, bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Bucket size')
axes[0].set_ylabel('Count')
axes[0].set_title(f'LSH Bucket Size Distribution (table 0, {len(lsh.tables[0])} buckets)')
axes[0].axvline(np.mean(bucket_sizes), color='red', linestyle='--', label=f'mean={np.mean(bucket_sizes):.1f}')
axes[0].legend()

# Show how number of candidates varies across random queries
n_candidates_list = []
for _ in range(200):
    q_test = np.random.randn(d).astype(np.float32)
    cands = set()
    for t in range(lsh.num_tables):
        key = lsh._hash(q_test, t)
        cands.update(lsh.tables[t].get(key, []))
    n_candidates_list.append(len(cands))

axes[1].hist(n_candidates_list, bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('Number of candidates')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Candidates per query (5 tables, 10 bits)')
axes[1].axvline(np.mean(n_candidates_list), color='red', linestyle='--',
                label=f'mean={np.mean(n_candidates_list):.0f}/{N}')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Product Quantization (PQ)

**Key idea**: Split each vector into M sub-vectors, quantize each sub-vector independently using k-means. Store only the centroid indices (compression). Approximate distance by summing sub-vector distances.

For a d-dimensional vector split into M sub-vectors, each quantized to K centroids:
- Storage: M * log2(K) bits per vector (vs 32*d bits for float32)
- Distance: precompute query-to-centroid distances, then sum lookup table entries

In [ ]:
class ProductQuantizer:
    """
    Product Quantization for vector compression and approximate distance.
    Uses simple k-means for each sub-quantizer.
    """
    def __init__(self, dim, M=8, K=256, n_iter=20):
        """
        dim: vector dimension (must be divisible by M)
        M: number of sub-quantizers
        K: number of centroids per sub-quantizer
        """
        assert dim % M == 0
        self.dim = dim
        self.M = M
        self.K = K
        self.dsub = dim // M
        self.n_iter = n_iter
        self.centroids = None  # (M, K, dsub)
        self.codes = None      # (N, M) uint8 indices
    
    def _kmeans(self, data, K):
        """Simple k-means on (N, dsub) data."""
        N = len(data)
        # Initialize centroids randomly
        idx = np.random.choice(N, K, replace=False)
        centroids = data[idx].copy()
        
        for _ in range(self.n_iter):
            # Assign
            dists = np.linalg.norm(data[:, None, :] - centroids[None, :, :], axis=2)  # (N, K)
            assignments = np.argmin(dists, axis=1)
            # Update
            for k in range(K):
                mask = assignments == k
                if mask.sum() > 0:
                    centroids[k] = data[mask].mean(axis=0)
        return centroids
    
    def train(self, data):
        """Train sub-quantizers on data (N, d)."""
        self.centroids = np.zeros((self.M, self.K, self.dsub), dtype=np.float32)
        for m in range(self.M):
            sub = data[:, m * self.dsub:(m + 1) * self.dsub]
            self.centroids[m] = self._kmeans(sub, self.K)
    
    def encode(self, data):
        """Encode vectors to PQ codes."""
        N = len(data)
        self.codes = np.zeros((N, self.M), dtype=np.uint8)
        for m in range(self.M):
            sub = data[:, m * self.dsub:(m + 1) * self.dsub]
            dists = np.linalg.norm(sub[:, None, :] - self.centroids[m][None, :, :], axis=2)
            self.codes[:, m] = np.argmin(dists, axis=1)
        return self.codes
    
    def search(self, query, k=5):
        """
        Asymmetric distance computation (ADC).
        Compute exact query-to-centroid distances, then approximate.
        """
        # Precompute distance table: (M, K)
        dist_table = np.zeros((self.M, self.K), dtype=np.float32)
        for m in range(self.M):
            q_sub = query[m * self.dsub:(m + 1) * self.dsub]
            dist_table[m] = np.linalg.norm(self.centroids[m] - q_sub[None, :], axis=1)
        
        # Approximate distances using lookup table
        N = len(self.codes)
        approx_dists = np.zeros(N, dtype=np.float32)
        for m in range(self.M):
            approx_dists += dist_table[m, self.codes[:, m]] ** 2
        approx_dists = np.sqrt(approx_dists)
        
        top_k = np.argsort(approx_dists)[:k]
        return top_k, approx_dists[top_k]


# Demo with small K for speed
pq = ProductQuantizer(dim=d, M=8, K=64, n_iter=10)
pq.train(db)
pq.encode(db)

pq_idx, pq_dists = pq.search(query, k=5)
exact_idx, exact_dists = brute_force_search(db, query, k=5, metric='euclidean')

print("PQ top-5:", pq_idx)
print("Exact top-5:", exact_idx)
print("Overlap:", len(set(pq_idx) & set(exact_idx)), "/ 5")
print(f"\nCompression: {d * 4} bytes -> {pq.M} bytes per vector ({d * 4 / pq.M:.0f}x)")

## 4. Approximate NN with Random Projections

**Johnson-Lindenstrauss lemma**: Random projections approximately preserve distances.

Project from d to d' << d, then do exact search in low-dimensional space.

In [ ]:
class RandomProjectionANN:
    """Approximate NN using random projection to lower dimension."""
    
    def __init__(self, dim, proj_dim=32):
        self.dim = dim
        self.proj_dim = proj_dim
        # Random projection matrix (scaled for distance preservation)
        self.projection = np.random.randn(proj_dim, dim).astype(np.float32) / np.sqrt(proj_dim)
        self.projected_data = None
    
    def index(self, data):
        """Project and store database vectors."""
        self.data = data
        self.projected_data = (self.projection @ data.T).T  # (N, proj_dim)
    
    def query(self, q, k=5):
        """Search in projected space, return original indices."""
        q_proj = self.projection @ q  # (proj_dim,)
        dists = np.linalg.norm(self.projected_data - q_proj[None, :], axis=1)
        top_k = np.argsort(dists)[:k]
        return top_k, dists[top_k]


rp_ann = RandomProjectionANN(dim=d, proj_dim=32)
rp_ann.index(db)

rp_idx, rp_dists = rp_ann.query(query, k=5)
print("Random Projection top-5:", rp_idx)
print("Exact (euclidean) top-5:", exact_idx)
print("Overlap:", len(set(rp_idx) & set(exact_idx)), "/ 5")

## 5. FAISS Patterns Overview

FAISS (Facebook AI Similarity Search) provides production-grade ANN implementations.

### Index Types and When to Use

| Index | How it works | When to use |
|-------|-------------|-------------|
| **Flat** | Brute force | < 100K vectors, need exact results |
| **IVF** (Inverted File) | Cluster with k-means, search only nearby clusters | 100K-10M vectors, moderate accuracy |
| **IVF + PQ** | IVF + product quantization | 1M-1B vectors, memory-constrained |
| **HNSW** (Hierarchical NSW) | Navigable small world graph | < 10M vectors, need high recall |
| **IVF + HNSW** | HNSW for coarse quantizer | Large scale with high recall |

### IVF Intuition
1. **Train**: Run k-means on training data to get `nlist` centroids
2. **Index**: Assign each vector to its nearest centroid
3. **Search**: Find `nprobe` nearest centroids to query, search only those cells
- Tradeoff: larger `nprobe` = better recall, slower search

### HNSW Intuition
1. Build a multi-layer graph (like a skip list)
2. Top layers: few nodes, long-range connections (for coarse navigation)
3. Bottom layers: all nodes, short-range connections (for fine search)
4. Search: start at top layer, greedily descend, then do beam search at bottom
- Key params: `M` (edges per node), `efConstruction` (beam width at build), `efSearch` (beam width at query)
- Tradeoff: more memory (graph edges), but excellent recall

### FAISS Index Selection Guide
```
if N < 100K:
    use IndexFlatL2 or IndexFlatIP  # exact, fast enough
elif N < 1M:
    use IndexHNSWFlat  # best recall, fits in memory
elif N < 100M:
    use IndexIVFFlat(nlist=sqrt(N))  # or IndexIVFPQ if memory-limited
else:  # billions
    use IndexIVFPQ or OPQ + IVF + PQ  # heavy compression needed
```

## 6. Benchmark: Exact vs Approximate on 10K Vectors

Compare speed and accuracy (recall@k) across methods.

In [ ]:
# Generate 10K database vectors
N_bench = 10000
d_bench = 128
db_bench = np.random.randn(N_bench, d_bench).astype(np.float32)

# Generate 100 query vectors
n_queries = 100
queries = np.random.randn(n_queries, d_bench).astype(np.float32)
k = 10

# Ground truth: brute force
print("Computing ground truth (brute force)...")
t0 = time.time()
gt_indices = []
for q in queries:
    idx, _ = brute_force_search(db_bench, q, k=k, metric='euclidean')
    gt_indices.append(set(idx))
bf_time = time.time() - t0
print(f"  Brute force: {bf_time:.3f}s for {n_queries} queries")

In [ ]:
def compute_recall(pred_indices_list, gt_indices_list, k):
    """Average recall@k."""
    recalls = []
    for pred, gt in zip(pred_indices_list, gt_indices_list):
        recalls.append(len(set(pred) & gt) / k)
    return np.mean(recalls)


# Benchmark LSH with different configurations
lsh_configs = [
    {'num_bits': 6, 'num_tables': 3},
    {'num_bits': 8, 'num_tables': 5},
    {'num_bits': 10, 'num_tables': 8},
    {'num_bits': 12, 'num_tables': 10},
    {'num_bits': 8, 'num_tables': 15},
]

results = []

# Brute force baseline
results.append({
    'method': 'Brute Force',
    'recall': 1.0,
    'time': bf_time,
    'qps': n_queries / bf_time
})

for cfg in lsh_configs:
    lsh_bench = LSHCosine(dim=d_bench, **cfg)
    lsh_bench.index(db_bench)
    
    t0 = time.time()
    pred_indices = []
    for q in queries:
        idx, _ = lsh_bench.query(q, k=k)
        pred_indices.append(list(idx))
    lsh_time = time.time() - t0
    
    # Note: LSH uses cosine, GT uses euclidean -- recall won't be perfect
    # For fair comparison, recompute GT with cosine
    gt_cos = []
    for q in queries:
        idx, _ = brute_force_search(db_bench, q, k=k, metric='cosine')
        gt_cos.append(set(idx))
    
    recall = compute_recall(pred_indices, gt_cos, k)
    label = f"LSH(bits={cfg['num_bits']},tables={cfg['num_tables']})"
    results.append({
        'method': label,
        'recall': recall,
        'time': lsh_time,
        'qps': n_queries / lsh_time
    })
    print(f"{label}: recall@{k}={recall:.3f}, time={lsh_time:.3f}s, QPS={n_queries/lsh_time:.0f}")

# Random Projection
for proj_dim in [16, 32, 64]:
    rp = RandomProjectionANN(dim=d_bench, proj_dim=proj_dim)
    rp.index(db_bench)
    
    t0 = time.time()
    pred_indices = []
    for q in queries:
        idx, _ = rp.query(q, k=k)
        pred_indices.append(list(idx))
    rp_time = time.time() - t0
    
    recall = compute_recall(pred_indices, gt_indices, k)
    label = f"RP(d'={proj_dim})"
    results.append({
        'method': label,
        'recall': recall,
        'time': rp_time,
        'qps': n_queries / rp_time
    })
    print(f"{label}: recall@{k}={recall:.3f}, time={rp_time:.3f}s, QPS={n_queries/rp_time:.0f}")

In [ ]:
# Visualize: Recall vs Speed
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

methods = [r['method'] for r in results]
recalls = [r['recall'] for r in results]
qps_vals = [r['qps'] for r in results]

# Scatter: recall vs QPS
colors = plt.cm.tab10(np.linspace(0, 1, len(results)))
for i, r in enumerate(results):
    axes[0].scatter(r['recall'], r['qps'], s=100, c=[colors[i]], zorder=5)
    axes[0].annotate(r['method'], (r['recall'], r['qps']),
                     textcoords='offset points', xytext=(5, 5), fontsize=7)

axes[0].set_xlabel('Recall@10')
axes[0].set_ylabel('Queries per second')
axes[0].set_title('Recall vs Speed Tradeoff')
axes[0].grid(True, alpha=0.3)

# Bar chart: recall comparison
x_pos = np.arange(len(results))
bars = axes[1].barh(x_pos, recalls, color=colors, edgecolor='black', alpha=0.8)
axes[1].set_yticks(x_pos)
axes[1].set_yticklabels(methods, fontsize=8)
axes[1].set_xlabel('Recall@10')
axes[1].set_title('Recall@10 by Method')
axes[1].set_xlim(0, 1.1)
for i, v in enumerate(recalls):
    axes[1].text(v + 0.01, i, f'{v:.2f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

## Interview Questions & Answers

### "How to do nearest neighbor search at scale?"
1. **Exact** is O(Nd) per query -- fine up to ~100K vectors.
2. **Approximate methods** trade accuracy for speed:
   - **LSH**: Hash-based, sub-linear query time, easy to distribute
   - **Quantization (PQ)**: Compress vectors, reduce memory and distance computation
   - **Graph-based (HNSW)**: Build navigable graph, best recall/speed tradeoff for moderate N
   - **IVF**: Cluster-based partitioning, combine with PQ for billions of vectors
3. In practice: use FAISS or ScaNN. Choose index type based on N, memory, and recall requirements.

### "LSH -- how does it work?"
- Design hash functions where P[h(a)=h(b)] is monotonically related to sim(a,b)
- For cosine: random hyperplane hashing. P[collision] = 1 - angle/pi
- Use multiple bits (AND amplification) to reduce false positives
- Use multiple tables (OR amplification) to reduce false negatives
- At query time: hash the query, collect candidates from matching buckets, re-rank

### "HNSW intuition?"
- Hierarchical graph inspired by skip lists
- Each layer is a proximity graph; higher layers have fewer nodes with longer-range edges
- Search: enter at top layer, greedily walk toward query, drop to next layer, repeat
- At bottom layer: do a beam search (ef neighbors) for fine-grained results
- Why it works: logarithmic number of layers means O(log N) hops to reach the right region

## Scale Considerations & Notes

| Scale | Approach | Notes |
|-------|----------|-------|
| < 100K | Brute force | Just use numpy or FAISS Flat |
| 100K - 1M | HNSW | Best recall, ~4-64 bytes/vector overhead for graph |
| 1M - 100M | IVF + PQ | Quantize to ~64 bytes/vector, partition for speed |
| > 100M | IVF + OPQ + PQ | Rotate before quantization (OPQ), distribute shards |
| > 1B | Distributed FAISS / custom | Shard across machines, use GPU for batch queries |

**Key tradeoffs**:
- Memory vs recall (PQ compression loses info)
- Build time vs query time (HNSW: slow build, fast query)
- Index params are tunable (nprobe, efSearch) -- always benchmark on your data